# VQE — Variational Quantum Eigensolver

**Objective key:** `vqe` &nbsp;·&nbsp; **Family:** Quantum &nbsp;·&nbsp; **Speed:** Compute-intensive

## Algorithm

A **parameterised quantum circuit** (ansatz) generates probability amplitudes. Its parameters  
are optimised by a classical outer loop (COBYLA, gradient-free) to minimise negative Sharpe ratio.

**Two execution paths:**

| Path | Ansatz | Solver | Requires |
|------|--------|--------|----------|
| Classical (default) | PauliTwoDesign (numpy simulation) | COBYLA | no hardware |
| IBM Quantum | EfficientSU2 | COBYLA + SamplerV2 on Qiskit Runtime | configured IBM token + `execution_kind: ibm_runtime` in run payload |

Marginal `|1⟩` probabilities per qubit → normalised portfolio weights.

## Papers

- **Foundational:** Peruzzo et al. (2014) — *A Variational Eigenvalue Solver on a Photonic Chip* — Nature Communications  
  https://arxiv.org/abs/1304.3061 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1304.3061
- **Modern:** Tilly et al. (2022) — *The Variational Quantum Eigensolver: A Review* — Physics Reports  
  https://arxiv.org/abs/2111.05176 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/2111.05176

## Assumptions

- `mu` and `Sigma` are annualised.
- `n_layers=2`, `n_restarts=3` for fast demo (production uses `n_layers=3`, `n_restarts=8`).
- IBM path is **not** exercised here; this notebook runs the classical simulation path only.
- `weight_min=0.005`, `weight_max=0.30`, `seed=42`.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api" / "app.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 6  # keep small: VQE circuit depth scales with n
mu = rng.uniform(0.07, 0.20, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"{n} assets (qubits in classical sim)")

In [ ]:
# Classical simulation path (no IBM token needed)
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="vqe",
    asset_names=asset_names,
    n_layers=2,       # circuit depth (PauliTwoDesign)
    n_restarts=3,     # COBYLA multi-start
    weight_min=0.005,
    weight_max=0.30,
    seed=42,
)

print(f"\nObjective:       {result.objective}")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nWeights:")
for name, w in zip(asset_names, result.weights):
    print(f"  {name}: {w:.4f}")

In [ ]:
# IBM Quantum path note
print("IBM Quantum path:")
print("  Requires: configured IBM token (POST /api/config/ibm-quantum)")
print("  Payload:  execution_kind='ibm_runtime', objective='vqe'")
print("  Ansatz:   EfficientSU2 (linear entanglement), transpiled via Qiskit")
print("  Sampler:  SamplerV2 on IBM Runtime (simulator or hardware backend)")
print("  Weights:  marginal |1> probabilities per qubit, clipped to [weight_min, weight_max]")
print(f"  Max qubits (hardware cap): {20}  (MAX_IBM_QUBITS in methods/vqe.py)")